<a href="https://colab.research.google.com/github/louisnguyen-eep/AgenticAIforBusiness118S/blob/dev/consildatedAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install anthropic langgraph langchain-core -q

import re
from datetime import datetime
from typing import Annotated
from typing_extensions import TypedDict

import anthropic
from google.colab import userdata

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage

# ── Claude Client ─────────────────────────────────────────────────────────────
api_key = userdata.get('claude-consolidated-agent')
client  = anthropic.Anthropic(api_key=api_key)
MODEL   = "claude-sonnet-4-5"

# ══════════════════════════════════════════════════════════════════════════════
# ── SHARED DATA ───────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

PRODUCTS = [
    {
        "name"      : "NexaPad Ultra",
        "category"  : "Tablet",
        "price"     : 899,
        "display"   : "12.9-inch Liquid Retina, 120 Hz",
        "chip"      : "Nexa A16 Bionic",
        "storage"   : "256 GB",
        "battery"   : "Up to 14 hours",
        "weight"    : "680 g",
        "extras"    : "Stylus support, keyboard cover compatible, 5G ready",
        "shipping"  : "2-day free shipping",
        "highlights": "The most powerful tablet NexaStore sells. Handles drawing, video editing, and note-taking effortlessly.",
        "best_for"  : "Digital artists, students, professionals who want a laptop replacement",
        "in_stock"  : True,
    },
    {
        "name"      : "NexaPad Lite",
        "category"  : "Tablet",
        "price"     : 449,
        "display"   : "10.2-inch IPS, 60 Hz",
        "chip"      : "Nexa A14",
        "storage"   : "128 GB",
        "battery"   : "Up to 10 hours",
        "weight"    : "490 g",
        "extras"    : "Wi-Fi only, stylus compatible, great for streaming",
        "shipping"  : "3-5 day standard shipping",
        "highlights": "Affordable and reliable everyday tablet. Perfect for browsing, streaming, and light work.",
        "best_for"  : "Casual users, kids, students on a budget",
        "in_stock"  : True,
    },
    {
        "name"      : "VisionWatch Pro",
        "category"  : "Smartwatch",
        "price"     : 399,
        "display"   : "1.9-inch Always-On AMOLED",
        "chip"      : "Nexa W3 health chip",
        "storage"   : "32 GB",
        "battery"   : "Up to 72 hours",
        "weight"    : "42 g",
        "extras"    : "ECG, blood oxygen, GPS, sleep tracking, 50m water resistance",
        "shipping"  : "2-day free shipping",
        "highlights": "Premium health and fitness tracker with an always-on display and 3-day battery life.",
        "best_for"  : "Fitness enthusiasts, health-conscious users, outdoor adventurers",
        "in_stock"  : True,
    },
    {
        "name"      : "VisionWatch SE",
        "category"  : "Smartwatch",
        "price"     : 199,
        "display"   : "1.7-inch AMOLED",
        "chip"      : "Nexa W2",
        "storage"   : "8 GB",
        "battery"   : "Up to 48 hours",
        "weight"    : "36 g",
        "extras"    : "Heart rate, step counter, sleep tracking, 30m water resistance",
        "shipping"  : "3-5 day standard shipping",
        "highlights": "Great entry-level smartwatch with solid health features at an accessible price.",
        "best_for"  : "First-time smartwatch buyers, casual fitness trackers",
        "in_stock"  : True,
    },
    {
        "name"      : "SoundDrop ANC",
        "category"  : "Wireless Earbuds",
        "price"     : 249,
        "display"   : "N/A",
        "chip"      : "Nexa H2 audio chip",
        "storage"   : "N/A",
        "battery"   : "8 hrs (buds) + 24 hrs (case)",
        "weight"    : "5.4 g per bud",
        "extras"    : "Active noise cancellation, transparency mode, wireless charging, IPX5",
        "shipping"  : "2-day free shipping",
        "highlights": "Studio-quality ANC earbuds with rich bass and crystal-clear calls.",
        "best_for"  : "Commuters, remote workers, music lovers who want noise cancellation",
        "in_stock"  : True,
    },
    {
        "name"      : "SoundDrop Go",
        "category"  : "Wireless Earbuds",
        "price"     : 99,
        "display"   : "N/A",
        "chip"      : "Nexa H1 audio chip",
        "storage"   : "N/A",
        "battery"   : "6 hrs (buds) + 18 hrs (case)",
        "weight"    : "4.8 g per bud",
        "extras"    : "Basic noise isolation, IPX4, fast pair",
        "shipping"  : "3-5 day standard shipping",
        "highlights": "Reliable everyday earbuds with great sound for the price.",
        "best_for"  : "Budget buyers, gym-goers, everyday listeners",
        "in_stock"  : True,
    },
    {
        "name"      : "NexaCam 4K",
        "category"  : "Action Camera",
        "price"     : 349,
        "display"   : "2.0-inch touchscreen",
        "chip"      : "Nexa GP5 imaging processor",
        "storage"   : "Up to 1 TB microSD",
        "battery"   : "Up to 2.5 hours recording",
        "weight"    : "132 g",
        "extras"    : "4K/60fps, waterproof to 10m, HyperSmooth stabilisation, voice control",
        "shipping"  : "3-5 day standard shipping",
        "highlights": "Rugged action camera that captures stunning 4K footage in any condition.",
        "best_for"  : "Hikers, surfers, cyclists, travel vloggers, adventure sports",
        "in_stock"  : True,
    },
    {
        "name"      : "DeskHub Pro",
        "category"  : "Smart Home Hub",
        "price"     : 179,
        "display"   : "7-inch HD touchscreen",
        "chip"      : "Nexa S4 smart chip",
        "storage"   : "16 GB",
        "battery"   : "Plugged in (2-hour backup)",
        "weight"    : "520 g",
        "extras"    : "Controls smart lights, locks, cameras; built-in voice assistant; Zigbee + Wi-Fi",
        "shipping"  : "3-5 day standard shipping",
        "highlights": "The central brain for your smart home — controls everything from one touchscreen.",
        "best_for"  : "Smart home enthusiasts, tech-savvy homeowners, families",
        "in_stock"  : True,
    },
    {
        "name"      : "ChargePad Trio",
        "category"  : "Wireless Charger",
        "price"     : 79,
        "display"   : "N/A",
        "chip"      : "N/A",
        "storage"   : "N/A",
        "battery"   : "N/A",
        "weight"    : "210 g",
        "extras"    : "Charges phone, earbuds, and smartwatch simultaneously; 15W fast charge; LED indicator",
        "shipping"  : "2-day free shipping",
        "highlights": "One pad to charge all your devices at once — no cable juggling.",
        "best_for"  : "Anyone with multiple NexaStore devices, minimalist desk setups",
        "in_stock"  : True,
    },
    {
        "name"      : "NexaLink Router",
        "category"  : "Wi-Fi Router",
        "price"     : 229,
        "display"   : "N/A",
        "chip"      : "Quad-core 1.8 GHz",
        "storage"   : "N/A",
        "battery"   : "Plugged in",
        "weight"    : "380 g",
        "extras"    : "Wi-Fi 6E, tri-band, covers up to 3,000 sq ft, parental controls, VPN support",
        "shipping"  : "3-5 day standard shipping",
        "highlights": "Blazing-fast Wi-Fi 6E router that eliminates dead zones and handles 100+ devices.",
        "best_for"  : "Gamers, large households, home office users, 4K streamers",
        "in_stock"  : True,
    },
]

ORDERS_DB = {
    "ORD-1001": {"status": "Shipped",          "item": "NexaPad Ultra",   "carrier": "FedEx", "tracking": "FX9284710234",          "estimated_date": "April 1, 2026"},
    "ORD-1002": {"status": "Processing",       "item": "VisionWatch Pro", "carrier": "UPS",   "tracking": None,                    "estimated_date": "April 4, 2026"},
    "ORD-1003": {"status": "Delivered",        "item": "SoundDrop ANC",   "carrier": "USPS",  "tracking": "9400111899223456789012","estimated_date": "March 25, 2026"},
    "ORD-1004": {"status": "Cancelled",        "item": "NexaLink Router", "carrier": "N/A",   "tracking": None,                    "estimated_date": "N/A"},
    "ORD-1005": {"status": "Out for Delivery", "item": "NexaCam 4K",      "carrier": "FedEx", "tracking": "FX1122334455",          "estimated_date": "March 29, 2026"},
    "ORD-1006": {"status": "Processing",       "item": "ChargePad Trio",  "carrier": "UPS",   "tracking": None,                    "estimated_date": "April 3, 2026"},
    "ORD-1007": {"status": "Shipped",          "item": "DeskHub Pro",     "carrier": "USPS",  "tracking": "9400111899223456789099","estimated_date": "April 2, 2026"},
    "ORD-1008": {"status": "Delivered",        "item": "NexaPad Lite",    "carrier": "FedEx", "tracking": "FX9988776655",          "estimated_date": "March 22, 2026"},
    "ORD-1009": {"status": "Shipped",          "item": "SoundDrop Go",    "carrier": "UPS",   "tracking": "1Z9999999999999999",    "estimated_date": "April 1, 2026"},
    "ORD-1010": {"status": "Processing",       "item": "VisionWatch SE",  "carrier": "FedEx", "tracking": None,                    "estimated_date": "April 5, 2026"},
}

REFUND_POLICY = {
    "window":        "30 days from the delivery date",
    "condition":     "Items must be unused, in original packaging, with all accessories included",
    "process_time":  "3–5 business days after the returned item is received",
    "method":        "Refund issued to the original payment method only",
    "exceptions":    "Opened software licenses and gift cards are non-refundable",
    "damaged_items": "If your item arrived damaged or defective, we cover return shipping and offer a full refund or replacement",
    "wrong_item":    "If you received the wrong item, contact us within 7 days and we will arrange a free return and re-ship",
    "how_to_start":  "Visit nexastore.com/returns, enter your order number, and follow the steps to print a return label",
    "contact":       "support@nexastore.com | Monday–Friday, 9 AM – 6 PM EST",
}

REFUND_CASES = {
    "REF-2001": {"status": "Approved",    "order": "ORD-1003", "item": "SoundDrop ANC",  "amount": "$249", "issued_date": "March 27, 2026"},
    "REF-2002": {"status": "Pending",     "order": "ORD-1008", "item": "NexaPad Lite",   "amount": "$449", "issued_date": "Awaiting item return"},
    "REF-2003": {"status": "Rejected",    "order": "ORD-1001", "item": "NexaPad Ultra",  "amount": "$0",   "issued_date": "N/A — outside return window"},
    "REF-2004": {"status": "In Progress", "order": "ORD-1009", "item": "SoundDrop Go",   "amount": "$99",  "issued_date": "Processing, 2–3 days remaining"},
    "REF-2005": {"status": "Approved",    "order": "ORD-1005", "item": "NexaCam 4K",     "amount": "$349", "issued_date": "March 30, 2026"},
    "REF-2006": {"status": "Pending",     "order": "ORD-1007", "item": "DeskHub Pro",    "amount": "$179", "issued_date": "Awaiting item return"},
}

# ══════════════════════════════════════════════════════════════════════════════
# ── SYSTEM PROMPTS ────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPTS = {
    "order": """
You are Alex, a friendly customer support agent for NexaStore, a premium online tech retailer.
Your only job is to help customers check their order status and shipping information.

BEHAVIOUR RULES:
- Be warm, concise, and professional.
- Use the exact order details provided in SYSTEM NOTEs — never invent information.
- If no order number is provided, politely ask for it (format: ORD-XXXX).
- If an order is not found, tell the customer and ask them to double-check.
- For delivered orders, confirm delivery and ask if everything arrived in good condition.
- For cancelled orders, empathise and direct them to support@nexastore.com if they need help.

COMPANY DETAILS:
- Support email : support@nexastore.com
- Support hours : Monday–Friday, 9 AM – 6 PM EST
- Website       : nexastore.com
""".strip(),

    "refund": """
You are Alex, a friendly customer support agent for NexaStore, a premium online tech retailer.
Your only job is to help customers with refund requests and return policy questions.

BEHAVIOUR RULES:
- Be warm, empathetic, concise, and professional.
- Always explain the relevant policy clearly before asking for more details.
- Use the exact refund case details provided in SYSTEM NOTEs — never invent information.
- If a customer asks about a specific refund case (REF-XXXX), use those details.
- If no refund case number is provided but they want to start a return, guide them to nexastore.com/returns.
- For damaged or wrong items, express extra empathy and assure them we will make it right.

COMPANY DETAILS:
- Support email : support@nexastore.com
- Support hours : Monday–Friday, 9 AM – 6 PM EST
- Return policy : 30-day hassle-free returns
- Website       : nexastore.com
""".strip(),

    "product": """
You are Sam, a knowledgeable and enthusiastic product specialist for NexaStore, a premium online tech retailer.
Your only job is to help customers find the perfect tech product.

RECOMMENDATION APPROACH:
- Ask about use case, budget, and must-have features before recommending.
- Recommend 1-2 products maximum per response to avoid overwhelming the customer.
- Explain WHY each product fits the customer's specific needs — personalise every recommendation.
- Never suggest products above the customer's budget without flagging the price difference.
- If a product is out of stock, say so and suggest the closest alternative.

BEHAVIOUR RULES:
- Be warm, enthusiastic, concise, and professional.
- Address the customer by name once you learn it.
- Never make up specs or prices — only use the product data provided in SYSTEM NOTEs.
- If a question requires account access or human help, say a specialist will follow up within 1 business day.

COMPANY DETAILS:
- Support email : support@nexastore.com
- Support hours : Monday–Friday, 9 AM – 6 PM EST
- Return policy : 30-day hassle-free returns
- Website       : nexastore.com
""".strip(),
}

# ══════════════════════════════════════════════════════════════════════════════
# ── ROUTER ────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def route_message(user_message: str) -> str:
    """Ask Claude which agent should handle this message."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=10,
        system="""You are a router for a customer service system at NexaStore, a tech retailer.
Classify the customer message into exactly one of these agents:

- order    : customer wants to track or check an order status
- refund   : customer wants a refund, return, has a damaged or wrong item, or asks about return policy
- product  : customer wants product recommendations, comparisons, specs, or help choosing a product

Reply with only one word: order, refund, or product.""",
        messages=[{"role": "user", "content": user_message}],
    )
    route = response.content[0].text.strip().lower()
    if route not in ("order", "refund", "product"):
        route = "product"  # safe default
    return route

# ══════════════════════════════════════════════════════════════════════════════
# ── CONTEXT BUILDERS ──────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

# ── Order ─────────────────────────────────────────────────────────────────────
def build_order_context(message: str) -> str:
    match = re.search(r'ORD-\d+', message, re.IGNORECASE)
    if not match:
        return "[SYSTEM NOTE: No order number found. Ask the customer for their order number (format: ORD-XXXX).]"
    order_id = match.group().upper()
    if order_id not in ORDERS_DB:
        return f"[SYSTEM NOTE: Order {order_id} was not found. Tell the customer and ask them to double-check.]"
    o        = ORDERS_DB[order_id]
    tracking = f"Tracking number: {o['tracking']}" if o['tracking'] else "Tracking number not yet assigned."
    return (
        f"[SYSTEM NOTE - Order details for {order_id}:\n"
        f"  Item: {o['item']} | Status: {o['status']} | Carrier: {o['carrier']}\n"
        f"  Estimated Delivery: {o['estimated_date']} | {tracking}\n"
        f"Use these exact details in your reply.]"
    )

# ── Refund ────────────────────────────────────────────────────────────────────
def detect_refund_intent(user_message: str) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=20,
        system="""Classify the customer message into exactly one of these intents:
check_case, damaged, wrong_item, how_to, policy, general

Reply with only the intent label, nothing else.""",
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text.strip().lower()

def build_refund_context(message: str) -> str:
    # Check for a specific REF case first
    match = re.search(r'REF-\d+', message, re.IGNORECASE)
    if match:
        case_id = match.group().upper()
        if case_id not in REFUND_CASES:
            return f"[SYSTEM NOTE: Refund case {case_id} was not found. Tell the customer and ask them to double-check.]"
        r = REFUND_CASES[case_id]
        return (
            f"[SYSTEM NOTE - Refund case details for {case_id}:\n"
            f"  Order: {r['order']} | Item: {r['item']} | Status: {r['status']} | "
            f"Amount: {r['amount']} | Issued/Expected: {r['issued_date']}\n"
            f"Use these exact details in your reply.]"
        )
    # Otherwise build policy context based on intent
    intent = detect_refund_intent(message)
    p      = REFUND_POLICY
    if intent == "damaged":
        note = p["damaged_items"]
    elif intent == "wrong_item":
        note = p["wrong_item"]
    elif intent == "how_to":
        note = f"How to start a return: {p['how_to_start']}"
    else:
        note = (
            f"Return window: {p['window']} | Condition: {p['condition']} | "
            f"Refund issued in: {p['process_time']} | Method: {p['method']} | "
            f"Exceptions: {p['exceptions']} | How to start: {p['how_to_start']}"
        )
    return f"[SYSTEM NOTE - Relevant refund policy:\n  {note}\n  Contact: {p['contact']}\nUse this in your reply.]"

# ── Product ───────────────────────────────────────────────────────────────────
def extract_budget(message: str) -> int | None:
    match = re.search(r'\$?\s*(\d{2,5})', message)
    return int(match.group(1)) if match else None

def detect_product_intent(user_message: str) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=20,
        system="""Classify the customer message into exactly one of these intents:
get_recommendation, compare, product_detail, stock_shipping, general

Reply with only the intent label, nothing else.""",
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text.strip().lower()

def build_product_context(message: str, budget: int | None) -> str:
    intent = detect_product_intent(message)

    if intent == "compare":
        lines = ["[SYSTEM NOTE - Full NexaStore product catalogue for comparison:"]
        for p in PRODUCTS:
            stock = "In Stock" if p["in_stock"] else "Out of Stock"
            lines.append(
                f"\n  {p['name']} — ${p['price']} [{stock}] | {p['category']}\n"
                f"    Chip: {p['chip']} | Storage: {p['storage']}\n"
                f"    Display: {p['display']} | Battery: {p['battery']} | Weight: {p['weight']}\n"
                f"    Extras: {p['extras']} | Best for: {p['best_for']}"
            )
        lines.append("\nCompare honestly based on the customer's needs.]")
        return "\n".join(lines)

    if intent == "product_detail":
        msg = message.lower()
        for p in PRODUCTS:
            if any(word in msg for word in p["name"].lower().split() if len(word) > 3):
                stock = "In Stock" if p["in_stock"] else "Out of Stock"
                return (
                    f"[SYSTEM NOTE - Full details for {p['name']} [{stock}]:\n"
                    f"  Price: ${p['price']} | Category: {p['category']}\n"
                    f"  Chip: {p['chip']} | Storage: {p['storage']}\n"
                    f"  Display: {p['display']} | Battery: {p['battery']} | Weight: {p['weight']}\n"
                    f"  Extras: {p['extras']} | Shipping: {p['shipping']}\n"
                    f"  Best for: {p['best_for']} | Summary: {p['highlights']}\n"
                    f"Answer the customer's specific question using only these details.]"
                )

    if intent == "stock_shipping":
        lines = ["[SYSTEM NOTE - Stock and shipping info:"]
        for p in PRODUCTS:
            lines.append(
                f"  {p['name']} ({p['category']}): "
                f"{'In Stock' if p['in_stock'] else 'Out of Stock'} | {p['shipping']}"
            )
        lines.append("]")
        return "\n".join(lines)

    # Default: recommendation catalogue filtered by budget
    eligible = [p for p in PRODUCTS if budget is None or p["price"] <= budget]
    if not eligible:
        eligible = PRODUCTS
    lines = ["[SYSTEM NOTE - Available NexaStore products:"]
    for p in eligible:
        stock = "In Stock" if p["in_stock"] else "Out of Stock"
        lines.append(
            f"\n  {p['name']} — ${p['price']} [{stock}] | {p['category']}\n"
            f"    Chip: {p['chip']} | Storage: {p['storage']}\n"
            f"    Display: {p['display']} | Battery: {p['battery']} | Weight: {p['weight']}\n"
            f"    Extras: {p['extras']} | Shipping: {p['shipping']}\n"
            f"    Best for: {p['best_for']} | Summary: {p['highlights']}"
        )
    if budget:
        lines.append(f"\n  Customer budget: ${budget} — only products within budget shown.")
    lines.append("\nRecommend 1-2 products max. Be conversational, not a spec dump.]")
    return "\n".join(lines)

# ══════════════════════════════════════════════════════════════════════════════
# ── LANGGRAPH STATE & NODES ───────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

class AgentState(TypedDict):
    messages       : Annotated[list, add_messages]
    active_agent   : str        # "order" | "refund" | "product"
    session_budget : int | None

def make_claude_node(agent_key: str):
    """Factory that returns a node function for a given agent."""
    def node(state: AgentState) -> dict:
        claude_messages = []
        for msg in state["messages"]:
            if isinstance(msg, HumanMessage):
                claude_messages.append({"role": "user",      "content": msg.content})
            elif isinstance(msg, AIMessage):
                claude_messages.append({"role": "assistant", "content": msg.content})

        response = client.messages.create(
            model=MODEL,
            max_tokens=512,
            system=SYSTEM_PROMPTS[agent_key],
            messages=claude_messages,
        )
        reply = response.content[0].text.strip()
        return {"messages": [AIMessage(content=reply)]}
    node.__name__ = f"{agent_key}_node"
    return node

def router_node(state: AgentState) -> dict:
    """Reads the last user message and sets active_agent."""
    last_human = next(
        (m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)),
        ""
    )
    agent = route_message(last_human)
    return {"active_agent": agent}

def pick_agent(state: AgentState) -> str:
    """Conditional edge: routes to the correct agent node."""
    return state["active_agent"]

# ── Build Graph ───────────────────────────────────────────────────────────────
def build_graph() -> StateGraph:
    memory  = MemorySaver()
    builder = StateGraph(AgentState)

    # Nodes
    builder.add_node("router",  router_node)
    builder.add_node("order",   make_claude_node("order"))
    builder.add_node("refund",  make_claude_node("refund"))
    builder.add_node("product", make_claude_node("product"))

    # Edges
    builder.add_edge(START, "router")
    builder.add_conditional_edges("router", pick_agent, {
        "order"  : "order",
        "refund" : "refund",
        "product": "product",
    })
    builder.add_edge("order",   END)
    builder.add_edge("refund",  END)
    builder.add_edge("product", END)

    return builder.compile(checkpointer=memory)

GRAPH = build_graph()

def invoke_graph(thread_id: str, human_content: str, session_budget: int | None,
                 active_agent: str = "product") -> tuple[str, str]:
    config = {"configurable": {"thread_id": thread_id}}
    result = GRAPH.invoke(
        {
            "messages"       : [HumanMessage(content=human_content)],
            "session_budget" : session_budget,
            "active_agent"   : active_agent,
        },
        config=config,
    )
    reply = result["messages"][-1].content
    agent = result["active_agent"]
    return reply, agent

# ══════════════════════════════════════════════════════════════════════════════
# ── MAIN CHAT SESSION ─────────────────────────────────════════════════════════
# ══════════════════════════════════════════════════════════════════════════════

AGENT_LABELS = {
    "order"  : "🚚 Order Agent   (Alex)",
    "refund" : "💸 Refund Agent  (Alex)",
    "product": "🛍️  Product Agent (Sam)",
}

def run_chat_session():
    thread_id      = datetime.now().strftime("%Y%m%d_%H%M%S")
    session_budget = None
    last_agent     = "product"

    print("=" * 60)
    print("  NexaStore — AI Customer Service")
    print(f"  Session ID (MemorySaver thread): {thread_id}")
    print("=" * 60)
    print("  Ask about orders, refunds, or products — I'll route you")
    print("  to the right agent automatically.")
    print("  Type 'done' to exit.")
    print("-" * 60)

    # Greeting (product agent handles it as a warm welcome)
    greeting, _ = invoke_graph(
        thread_id,
        "Greet the customer warmly. Introduce NexaStore and let them know you can "
        "help with orders, refunds, and product recommendations.",
        session_budget,
    )
    print(f"\n  Alex: {greeting}\n")

    # Conversation loop
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("done", "quit", "exit", "bye"):
            break

        # Update budget if mentioned
        found_budget = extract_budget(user_input)
        if found_budget:
            session_budget = found_budget

        # Build the right context BEFORE invoking the graph
        # (router runs first, but we pre-build context using the same routing logic)
        agent_key = route_message(user_input)

        if agent_key == "order":
            context = build_order_context(user_input)
        elif agent_key == "refund":
            context = build_refund_context(user_input)
        else:
            context = build_product_context(user_input, session_budget)

        augmented  = f"{user_input}\n\n{context}"
        reply, last_agent = invoke_graph(thread_id, augmented, session_budget, agent_key)

        print(f"\n  [{AGENT_LABELS[last_agent]}]")
        print(f"  {reply}\n")
        print("-" * 60)

    # Closing
    closing, _ = invoke_graph(
        thread_id,
        "The customer is leaving. Give a warm one-sentence goodbye.",
        session_budget,
        last_agent,
    )
    print(f"\n  {closing}\n")
    print("=" * 60)

    snapshot  = GRAPH.get_state({"configurable": {"thread_id": thread_id}})
    msg_count = len(snapshot.values["messages"])
    print(f"\n  [MemorySaver] {msg_count} messages checkpointed for thread '{thread_id}'")

# ── Run ───────────────────────────────────────────────────────────────────────
run_chat_session()

# How the routing works:
# The graph now has 4 nodes connected like this:
# START → router → (conditional edge) → order   → END
#                                     → refund  → END
#                                     → product → END

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 7.9 MB/s eta 0:00:00
  NexaStore — AI Customer Service
  Session ID (MemorySaver thread): 20260330_010937
  Ask about orders, refunds, or products — I'll route you
  to the right agent automatically.
  Type 'done' to exit.
------------------------------------------------------------

  Alex: Hey there! 👋 Welcome to **NexaStore** — your go-to destination for premium tech that actually makes sense for your life.

I'm Sam, and I'm here to help you find exactly what you need, whether that's:

✅ **Product recommendations** — laptops, headphones, smart home gear, you name it  
✅ **Order questions** — tracking, updates, or changes  
✅ **Returns & refunds** — we've got a hassle-free 30-day return policy  

Just let me know what you're looking for or what's on your mind, and I'll take it from here. What can I help you with today? 😊

You: done

  Thanks for stopping by — feel free to reach out anytime, and happy tech shopping! 🚀


  [Mem